# Impact-Screened Quant Strategy
### A transparent, values-first equity ranking & backtest engine

This notebook builds a stock-selection strategy in three honest layers:

1. **Values screen** — you invest *only* in companies on a theme list **you** control (clean energy, healthcare, scientific tools, etc.). "Advancing humanity" is a judgment, so the list is yours to edit, not something this code decides for you.
2. **Factor analytics** — within that screened universe it scores every name on **momentum, trend, quality (low-vol), and a valuation sanity check**, so it favors high performers *without* blindly chasing the most overbought ones.
3. **Honest backtest** — it walks the strategy forward month by month (no lookahead), sizes positions by inverse volatility with caps, applies a market-regime filter, and reports returns, drawdowns, and turnover **against a simple buy-and-hold benchmark**.

> **Read this before trusting any number below.** This is research tooling, not financial advice, and not a promise of returns. A backtest is a hypothesis about the past, not a guarantee about the future. The most useful thing this notebook can tell you is *when the strategy does **not** beat just holding an index* — and it will tell you that plainly. Decisions are yours.


In [ ]:
# If running in Colab or a fresh environment, uncomment:
# !pip install yfinance pandas numpy matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings; warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})
print("Environment ready.")

## 1 · Your values universe  ✏️ *edit this*

Below is a **starting** map of themes commonly associated with positive impact, with example tickers. **This is the heart of the strategy and it encodes a values judgment — make it yours.** Add names you believe advance humanity, delete ones you disagree with, add whole new themes.

If you'd rather screen a *broad* market and merely **exclude** harmful industries (tobacco, weapons, fossil fuels, gambling), there's a note at the bottom of the notebook on switching to that approach.

In [ ]:
# ── EDIT FREELY ──────────────────────────────────────────────────────────
IMPACT_UNIVERSE = {
    "Clean energy & storage": ["ENPH", "FSLR", "NEE", "BEP", "SEDG", "RUN"],
    "Sustainable transport":  ["TSLA", "RIVN", "ALB"],
    "Healthcare & cures":     ["VRTX", "REGN", "ISRG", "DHR", "TMO", "ILMN"],
    "Scientific tools":       ["A", "MTD", "WAT"],
    "Enabling semiconductors":["NVDA", "TSM", "ASML", "AMD"],
    "Water & resources":      ["XYL", "AWK", "ECL"],
    "Food & ag innovation":   ["DE", "CTVA"],
    "Access & education":     ["DUOL", "COUR"],
}

# Flatten to a clean ticker list
TICKERS = sorted({t for names in IMPACT_UNIVERSE.values() for t in names})
BENCHMARK = "SPY"   # buy-and-hold yardstick
print(f"{len(TICKERS)} tickers across {len(IMPACT_UNIVERSE)} themes:")
print(", ".join(TICKERS))

## 2 · Strategy parameters  ✏️ *tune this*

Every knob is exposed. The factor **weights** decide the personality of the strategy: raise `momentum`/`trend` to chase strength, raise `value` to lean against overbought names, raise `quality` to prefer calmer stocks.

In [ ]:
CONFIG = {
    "lookback_years": 8,        # history to download
    "top_n": 8,                 # holdings per rebalance
    "max_weight": 0.20,         # cap per position
    "rebalance": "ME",          # month-end
    "use_regime_filter": True,  # go to cash when SPY < its 200d MA
    "cost_bps": 10,             # round-trip transaction cost, basis points
    "weights": {                # factor blend (need not sum to 1)
        "momentum": 0.35,
        "trend":    0.25,
        "quality":  0.20,
        "value":    0.20,
    },
    # analytics windows
    "mom_lookback": 252, "mom_skip": 21, "vol_lookback": 63, "trend_window": 200,
}
CONFIG

## 3 · Download data

Free daily prices via `yfinance`. If a ticker fails or is too new, it's dropped with a note rather than crashing the run.

In [ ]:
def download_prices(tickers, years, benchmark):
    end = pd.Timestamp.today()
    start = end - pd.DateOffset(years=years)
    raw = yf.download(tickers + [benchmark], start=start, end=end,
                      auto_adjust=True, progress=False)["Close"]
    raw = raw.dropna(how="all")
    # drop tickers with too little history
    good = [c for c in raw.columns if raw[c].notna().sum() > 300]
    dropped = [c for c in raw.columns if c not in good]
    if dropped:
        print("Dropped (insufficient history):", ", ".join(dropped))
    return raw[good]

prices_all = download_prices(TICKERS, CONFIG["lookback_years"], BENCHMARK)
bench = prices_all[BENCHMARK].dropna()
prices = prices_all[[c for c in prices_all.columns if c != BENCHMARK]]
print(f"\nLoaded {prices.shape[1]} names, {prices.shape[0]} trading days "
      f"({prices.index[0].date()} → {prices.index[-1].date()})")
prices.tail(3)

## 4 · The analytics engine

Pure functions, each computed **only from data available at the decision date** — this is what keeps the backtest from cheating with future information.

- **Momentum (12-1):** total return from ~12 months ago to ~1 month ago (skips the most recent month to avoid short-term reversal).
- **Trend:** price relative to its 200-day moving average.
- **Quality:** *inverse* of realized volatility — calmer compounders score higher.
- **Value sanity:** an RSI-based check that favors names near neutral and **penalizes the overbought**, so "high performer" doesn't collapse into "buy the top."

Each factor is converted to a cross-sectional **z-score** (standardized within the universe), then blended by your weights into one `composite` score.

In [ ]:
def _zscore(s):
    s = s.astype(float); sd = s.std(ddof=0)
    return pd.Series(0.0, index=s.index) if (sd == 0 or np.isnan(sd)) else (s - s.mean()) / sd

def _winsorize(z, k=3.0):
    # clip standardized factor at +/- k sigma — standard quant hygiene so one
    # outlier name can't dominate the composite (Barra/AQR-style robustness).
    return z.clip(-k, k)

def rsi(px, window=14):
    d = px.diff()
    up = d.clip(lower=0).rolling(window).mean()
    dn = (-d.clip(upper=0)).rolling(window).mean()
    rs = up / dn.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

def compute_scores(prices, asof, cfg):
    w = cfg["weights"]
    px = prices.loc[:asof]
    cols = [c for c in px.columns if px[c].notna().sum() > cfg["mom_lookback"] + 5]
    px = px[cols]
    if px.shape[1] == 0: return pd.DataFrame()
    last = px.iloc[-1]
    momentum = (px.shift(cfg["mom_skip"]).iloc[-1] / px.shift(cfg["mom_lookback"]).iloc[-1]) - 1
    trend = (last / px.rolling(cfg["trend_window"]).mean().iloc[-1]) - 1
    vol = px.pct_change().iloc[-cfg["vol_lookback"]:].std() * np.sqrt(252)
    r = rsi(px).iloc[-1]
    good_price = -(r - 50).abs() / 50
    overbought = (r.clip(lower=60) - 60) / 40

    df = pd.DataFrame({"price": last, "momentum": momentum, "trend": trend,
                       "volatility": vol, "rsi": r})
    # winsorized cross-sectional z-scores
    df["z_momentum"] = _winsorize(_zscore(df["momentum"]))
    df["z_trend"]    = _winsorize(_zscore(df["trend"]))
    df["z_quality"]  = _winsorize(_zscore(-df["volatility"]))
    df["z_value"]    = _winsorize(_zscore(good_price) - _zscore(overbought))
    df["composite"]  = (w["momentum"]*df.z_momentum + w["trend"]*df.z_trend +
                        w["quality"]*df.z_quality + w["value"]*df.z_value)
    return df.sort_values("composite", ascending=False)

print("Analytics engine defined (winsorized factor z-scores).")

## 5 · Today's ranked picks

What the strategy would favor **right now**, with the factor breakdown so you can see *why* each name ranks where it does — never trust a score you can't decompose.

In [ ]:
today = prices.index[-1]
scores = compute_scores(prices, today, CONFIG)

# attach theme labels
ticker_theme = {t: theme for theme, names in IMPACT_UNIVERSE.items() for t in names}
scores["theme"] = [ticker_theme.get(t, "") for t in scores.index]

show = scores[["theme","price","momentum","trend","volatility","rsi",
               "z_momentum","z_trend","z_quality","z_value","composite"]]
print(f"Ranked universe as of {today.date()}:\n")
show.round(3)

In [ ]:
top = scores.head(CONFIG["top_n"])
print(f"TOP {CONFIG['top_n']} CANDIDATES (the strategy's current shortlist)\n")
for i, (tk, row) in enumerate(top.iterrows(), 1):
    tag = "good entry" if row.rsi < 55 else ("extended" if row.rsi > 70 else "neutral")
    print(f"{i:>2}. {tk:<5} {row.theme:<26} composite {row.composite:+.2f} "
          f"| RSI {row.rsi:4.0f} ({tag})")

## 6 · Action grades — *what the model says to do*

This turns the scores into a plain verdict per name: a letter grade (A–F by rank within your screen) and an action (**BUY / BUY ON DIP / HOLD / UNDERWEIGHT / AVOID**), each with a one-line reason. It's **regime-aware** — when the market is risk-off (index below its 200-day average) it defaults everything defensive.

> These are **mechanical outputs of the model's ranking**, conditioned on the name already passing *your* values screen. A grade of "A / BUY" means "ranks top of your universe on momentum, trend, quality and valuation right now" — not a recommendation. The model has no idea what you can afford, your taxes, or your risk tolerance. You decide.

In [ ]:
def grade_universe(scores, bench, prices, asof, cfg):
    """Mechanical grade + action per name. Not advice — model output only."""
    risk_on = regime_ok(bench, asof)
    s = scores.copy()
    s["pct"] = s["composite"].rank(pct=True)
    top_set = set(s.head(cfg["top_n"]).index)   # names the portfolio would actually hold
    rows = []
    for tk, r in s.iterrows():
        p, rsi = r["pct"], r["rsi"]
        letter = "A" if p>=0.8 else "B" if p>=0.6 else "C" if p>=0.4 else "D" if p>=0.2 else "F"
        overbought, oversold = rsi>70, rsi<35
        in_top = tk in top_set
        if not risk_on:
            action, note = "HOLD / RAISE CASH", "Market risk-off (index < 200d MA); model defaults defensive."
        elif in_top and not overbought:
            action, note = "BUY", "In the portfolio's top picks and not overbought — clean entry."
        elif in_top and overbought:
            action, note = "BUY ON DIP", f"A top pick but RSI {rsi:.0f} (extended) — wait for a pullback."
        elif p>=0.5:
            action, note = "HOLD", "Mid-pack — keep if owned, no strong add signal."
        elif p>=0.25:
            action, note = "UNDERWEIGHT", "Lower half of your screen on current factors."
        else:
            action, note = "AVOID", "Bottom-ranked on momentum/trend/quality/value now."
        if oversold and not in_top:
            note += " RSI oversold — watch for reversal (unconfirmed)."
        rows.append({"grade":letter, "action":action, "in_portfolio":in_top,
                     "composite":round(r["composite"],2), "rank_%":int(round(p*100)),
                     "rsi":int(round(rsi)), "theme":ticker_theme.get(tk,""), "why":note})
    return pd.DataFrame(rows, index=s.index), risk_on

grades, risk_on = grade_universe(scores, bench, prices, today, CONFIG)
print(f"MARKET REGIME: {'RISK-ON (index above 200d MA)' if risk_on else 'RISK-OFF (index below 200d MA) — model defensive'}\n")
order = {"BUY":0,"BUY ON DIP":1,"HOLD":2,"UNDERWEIGHT":3,"AVOID":4,"HOLD / RAISE CASH":5}
grades_sorted = grades.sort_values(by="action", key=lambda c: c.map(order))
grades_sorted[["grade","action","theme","rank_%","rsi","why"]]

In [ ]:
# Optional color-coded view (renders in Jupyter/Colab; falls back to plain text)
def _color(v):
    m = {"BUY":"#1b7f4b","BUY ON DIP":"#2a9d8f","HOLD":"#b08900",
         "UNDERWEIGHT":"#c0631f","AVOID":"#b3261e","HOLD / RAISE CASH":"#5b6b7a"}
    return f"background-color:{m.get(v,'#444')};color:white;font-weight:600"
try:
    disp = grades_sorted[["grade","action","theme","rank_%","rsi"]].copy()
    display(disp.style.applymap(_color, subset=["action"])
                 .set_caption("Action grades — model output, not advice"))
except Exception as e:
    print(grades_sorted[["grade","action","theme","rank_%","rsi"]].to_string())

### Your trade list — *target vs. what you already hold*

Tell it what you currently own (as portfolio weights) and your capital, and it prints the exact **OPEN / ADD / TRIM / CLOSE** moves to reach the target allocation. Leave `CURRENT_HOLDINGS` empty if you're starting from scratch.

*Mechanical rebalancing math, not a recommendation to trade.*

In [ ]:
# ✏️ what you own now, as weights (e.g. {"NVDA": 0.15, "TSLA": 0.10}); {} if none
CURRENT_HOLDINGS = {}
CAPITAL = 10_000   # set to your investable amount for dollar deltas, or None

def trade_list(target_w, holdings, capital=None):
    cur = pd.Series(holdings, dtype=float)
    rows = {}
    for tk in sorted(set(target_w.index) | set(cur.index)):
        tw, cw = float(target_w.get(tk,0)), float(cur.get(tk,0)); d = tw-cw
        act = ("HOLD" if abs(d)<0.01 else "OPEN" if cw==0 else
               "CLOSE" if tw==0 else "ADD" if d>0 else "TRIM")
        r = {"action":act, "current_%":round(cw*100,1), "target_%":round(tw*100,1),
             "delta_%":round(d*100,1)}
        if capital: r["delta_$"] = round(d*capital,2)
        rows[tk] = r
    out = pd.DataFrame(rows).T
    ordr = {"OPEN":0,"ADD":1,"TRIM":2,"CLOSE":3,"HOLD":4}
    return out.sort_values("action", key=lambda c: c.map(ordr))

orders = trade_list(target, CURRENT_HOLDINGS, CAPITAL)
print("TRADE LIST to reach target allocation\n")
orders

## 7 · Portfolio construction

Inverse-volatility weighting (calmer names get more capital), each position capped, weights renormalized to 100%.

In [ ]:
def build_weights(scores, prices, asof, cfg):
    picks = scores.head(cfg["top_n"]).index.tolist()
    if not picks: return pd.Series(dtype=float)
    vol = prices[picks].loc[:asof].pct_change().iloc[-cfg["vol_lookback"]:].std() * np.sqrt(252)
    inv = (1 / vol.replace(0, np.nan)); inv = inv.fillna(inv.mean())
    w = inv / inv.sum()
    for _ in range(10):
        over = w > cfg["max_weight"]
        if not over.any(): break
        excess = (w[over] - cfg["max_weight"]).sum(); w[over] = cfg["max_weight"]
        under = ~over
        if under.any(): w[under] += excess * (w[under] / w[under].sum())
    return (w / w.sum()).sort_values(ascending=False)

target = build_weights(scores, prices, today, CONFIG)
print("TARGET ALLOCATION\n")
for tk, wt in target.items():
    print(f"  {tk:<5} {wt*100:5.1f}%   {ticker_theme.get(tk,'')}")
print(f"\n  total {target.sum()*100:.1f}%")

plt.bar(target.index, target.values*100, color="#2a9d8f")
plt.ylabel("weight %"); plt.title("Current target allocation"); plt.show()

## 8 · Walk-forward backtest

Each month: score the universe using only past data → pick top *N* → weight them → hold for the next month → record the return. With `use_regime_filter` on, the strategy sits in **cash** whenever SPY closes below its 200-day average (a crude but effective bear-market dodge).

This is the honest part. Watch how it compares to simply holding SPY.

In [ ]:
def regime_ok(bench, asof, ma=200):
    b = bench.loc[:asof].dropna()
    return True if len(b) < ma + 1 else b.iloc[-1] >= b.rolling(ma).mean().iloc[-1]

def backtest(prices, bench, cfg, start=None):
    """Walk-forward, monthly. Subtracts transaction costs on the turned-over fraction."""
    cost = cfg.get("cost_bps", 0) / 10000.0
    me = prices.resample(cfg["rebalance"]).last().index
    if start: me = me[me >= pd.Timestamp(start)]
    eq, curve, dates, prev, turns = 1.0, [], [], None, []
    for i in range(len(me) - 1):
        asof, nxt = me[i], me[i+1]
        if prices.loc[:asof].shape[0] < 260: continue
        if cfg["use_regime_filter"] and not regime_ok(bench, asof):
            curve.append(eq); dates.append(nxt); prev = None; continue
        sc = compute_scores(prices, asof, cfg)
        if sc.empty: curve.append(eq); dates.append(nxt); continue
        w = build_weights(sc, prices, asof, cfg)
        tno = w.subtract(prev, fill_value=0).abs().sum()/2 if prev is not None else 1.0
        turns.append(tno); prev = w
        fwd = prices[w.index].loc[asof:nxt].pct_change().iloc[1:]
        gross = (fwd @ w).add(1).prod() - 1
        eq *= (1 + gross - tno * 2 * cost)   # round-trip cost on turnover
        curve.append(eq); dates.append(nxt)
    return pd.Series(curve, index=pd.DatetimeIndex(dates)), (np.mean(turns) if turns else 0)

start_date = (prices.index[0] + pd.DateOffset(months=14)).strftime("%Y-%m-%d")
curve, avg_turnover = backtest(prices, bench, CONFIG, start=start_date)
bench_m = bench.resample(CONFIG["rebalance"]).last().reindex(curve.index).ffill()
bench_curve = bench_m / bench_m.iloc[0]
print(f"Backtest {curve.index[0].date()} -> {curve.index[-1].date()} "
      f"({len(curve)} months) | avg monthly turnover {avg_turnover*100:.0f}% "
      f"| costs {CONFIG['cost_bps']}bp/trade")

In [ ]:
def metrics(curve, freq=12):
    r = curve.pct_change().dropna()
    if len(r) < 2: return {}
    yrs = (curve.index[-1]-curve.index[0]).days/365.25
    dd = curve/curve.cummax()-1
    vol = r.std()*np.sqrt(freq); dn = r[r<0].std()*np.sqrt(freq)
    return {"CAGR": curve.iloc[-1]**(1/yrs)-1, "Vol": vol,
            "Sharpe": (r.mean()*freq)/vol if vol else np.nan,
            "Sortino": (r.mean()*freq)/dn if dn else np.nan,
            "MaxDD": dd.min(), "HitRate": (r>0).mean()}

ms, mb = metrics(curve), metrics(bench_curve)
comp = pd.DataFrame({"Strategy": ms, f"Buy & Hold {BENCHMARK}": mb}).T
comp["CAGR"] = comp["CAGR"].map(lambda x: f"{x*100:.1f}%")
comp["Vol"] = comp["Vol"].map(lambda x: f"{x*100:.1f}%")
comp["MaxDD"] = comp["MaxDD"].map(lambda x: f"{x*100:.1f}%")
comp["HitRate"] = comp["HitRate"].map(lambda x: f"{x*100:.0f}%")
comp[["Sharpe","Sortino"]] = comp[["Sharpe","Sortino"]].round(2)
print("PERFORMANCE — strategy vs simply holding the index\n")
comp[["CAGR","Vol","Sharpe","Sortino","MaxDD","HitRate"]]

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), height_ratios=[2.5, 1], sharex=True)
ax1.plot(curve.index, curve.values, label="Impact strategy", color="#2a9d8f", lw=2)
ax1.plot(bench_curve.index, bench_curve.values, label=f"Buy & hold {BENCHMARK}",
         color="#9aa6b2", lw=1.6, ls="--")
ax1.set_yscale("log"); ax1.set_ylabel("growth of $1 (log)")
ax1.set_title("Walk-forward equity curve"); ax1.legend()
dd = curve/curve.cummax()-1
ax2.fill_between(dd.index, dd.values*100, 0, color="#e76f51", alpha=0.5)
ax2.set_ylabel("drawdown %"); ax2.set_title("Strategy drawdowns")
plt.tight_layout(); plt.show()

## 9 · Research-grade diagnostics — *am I fooling myself?*

A serious quant spends most of their time here, not on the strategy. Three checks, each grounded in published research:

**(a) Information Coefficient (IC).** The honest test of a factor is whether it actually predicts: the rank correlation between a factor's score today and next month's returns. Mean IC near zero = noise, no matter how good the backtest looks. We also report the **IC information ratio** (mean ÷ volatility) — a factor can have a tiny edge that's swamped by how unstable it is.

**(b) Costs & decay.** Backtests are optimistic for two well-documented reasons:
- *Transaction costs* (already subtracted above via the turnover figure).
- *Publication decay.* McLean & Pontiff (2016, J. Finance) found published predictors earned **26% less out-of-sample and 58% less after publication** — investors learn and arbitrage them away. Every factor here is public, so we haircut the excess return by those figures to show a sober expectation, not the rosy backtest.

**(c) Multiple-testing significance.** Harvey, Liu & Zhu (2016, RFS) showed that with hundreds of factors mined, the usual *t* > 2 bar is far too lenient — a credible factor needs roughly **t > 3**. We compute the strategy's t-stat and judge it against that higher bar.

> The point of this section is to *try to kill the strategy*. If it survives honest costs, a publication-decay haircut, and a t > 3 hurdle, it's worth a second look. If it doesn't — which is the common outcome — you've learned something real and cheap.

In [ ]:
def factor_ic(prices, cfg, factor_cols, start=None):
    """Spearman rank IC between each factor at t and forward 1-month return."""
    pme = prices.resample(cfg["rebalance"]).last(); me = pme.index
    if start: me = me[me >= pd.Timestamp(start)]
    recs = {f: [] for f in factor_cols}
    for i in range(len(me) - 1):
        asof, nxt = me[i], me[i+1]
        if prices.loc[:asof].shape[0] < 260: continue
        sc = compute_scores(prices, asof, cfg)
        if sc.empty: continue
        fwd = (pme.loc[nxt] / pme.loc[asof] - 1).reindex(sc.index)
        ok = fwd.notna()
        if ok.sum() > 5:
            for f in factor_cols:
                recs[f].append(sc.loc[ok, f].corr(fwd[ok], method="spearman"))
    rows = {}
    for f, v in recs.items():
        v = pd.Series(v).dropna()
        rows[f] = {"mean_IC": v.mean(), "IC_std": v.std(),
                   "IC_IR": v.mean()/v.std() if v.std() else np.nan,
                   "hit_%": (v > 0).mean()*100, "months": len(v)}
    return pd.DataFrame(rows).T

ic = factor_ic(prices, CONFIG, ["z_momentum","z_trend","z_quality","z_value"], start=start_date)
print("INFORMATION COEFFICIENT — does each factor actually predict forward returns?\n")
print("(mean_IC > ~0.03 with positive IC_IR is a faint but real signal; ~0 is noise)\n")
ic.round(3)

In [ ]:
# Cost-adjusted vs decay-adjusted expectations
def cagr(c):
    yrs = (c.index[-1]-c.index[0]).days/365.25
    return c.iloc[-1]**(1/yrs) - 1

strat_cagr, bench_cagr = cagr(curve), cagr(bench_curve)
excess = strat_cagr - bench_cagr
decay = {"In-sample excess (backtest)": excess,
         "Out-of-sample (-26%, McLean-Pontiff)": excess*(1-0.26),
         "Post-publication (-58%, McLean-Pontiff)": excess*(1-0.58)}

print(f"Strategy CAGR {strat_cagr*100:5.1f}%   Benchmark CAGR {bench_cagr*100:5.1f}%   "
      f"Excess {excess*100:+.1f}%\n")
print("REALITY-DISCOUNTED excess return over benchmark (these factors are all public):")
for k, v in decay.items():
    print(f"  {k:<42} {v*100:+5.1f}%")

# Multiple-testing significance check
r = curve.pct_change().dropna()
t_stat = r.mean() / (r.std()/np.sqrt(len(r)))
print(f"\nMonthly return t-statistic: {t_stat:.2f}")
print(f"  clears conventional t>2.0 ?  {'YES' if abs(t_stat)>2 else 'NO'}")
print(f"  clears Harvey-Liu-Zhu t>3.0? {'YES' if abs(t_stat)>3 else 'NO'}  "
      f"(the bar that accounts for hundreds of factors being mined)")

## 10 · Monte Carlo — *your backtest is one path out of thousands*

The equity curve above is a single history. Monte Carlo asks: given how this strategy actually behaves month to month, what's the **full range** of outcomes it could have produced — and could produce going forward? This converts the vague "past ≠ future" caveat into an actual probability distribution.

Two honest design choices:

- **Block bootstrap, not IID.** We resample *blocks* of consecutive months, not single months. Momentum strategies have serial correlation — good runs and bad runs cluster. Naive IID resampling shatters that and makes the strategy look *safer* than it is. The block version preserves the clustering, so drawdowns and loss probabilities come out realistically worse. (You can switch `method="iid"` to see how much the naive version flatters the strategy.)
- **A decay-adjusted forward cone.** A second simulation recenters returns to the publication-decay expectation from Section 9 (these factors are public), keeping the volatility but sobering the mean. It's labelled a *scenario*, not a forecast — the point is to see the risk, not to predict a number.

What to read: the **probability of a net loss**, the **5th-percentile (bad-luck) outcome**, and the **probability of beating buy-and-hold**. If the strategy only wins in the median case but coin-flips against the benchmark across simulations, that's a fragile edge.

In [ ]:
def monte_carlo(returns, n_sims=5000, horizon=None, block=6, method="block", seed=42):
    """Bootstrap simulated equity paths from realized periodic returns.
       method='block' preserves momentum autocorrelation; 'iid' assumes independence."""
    r = pd.Series(returns).dropna().values
    if len(r) < block + 1: block = max(1, len(r) // 4)
    if horizon is None: horizon = len(r)
    rng = np.random.default_rng(seed)
    paths = np.empty((n_sims, horizon + 1)); paths[:, 0] = 1.0
    if method == "iid":
        samp = rng.choice(r, size=(n_sims, horizon), replace=True)
    else:
        nblocks = int(np.ceil(horizon / block)); max_start = max(1, len(r) - block + 1)
        samp = np.empty((n_sims, nblocks * block))
        for i in range(n_sims):
            starts = rng.integers(0, max_start, size=nblocks)
            samp[i] = np.concatenate([r[s:s + block] for s in starts])
        samp = samp[:, :horizon]
    paths[:, 1:] = np.cumprod(1 + samp, axis=1)
    return paths

def mc_summary(paths, ppy=12, benchmark_mult=None):
    term = paths[:, -1]; years = (paths.shape[1] - 1) / ppy
    cagr = term ** (1 / years) - 1
    runmax = np.maximum.accumulate(paths, axis=1); dd = (paths / runmax - 1).min(axis=1)
    out = {"Terminal x (median)": np.median(term), "Terminal x (5th pct)": np.percentile(term, 5),
           "Terminal x (95th pct)": np.percentile(term, 95), "CAGR median": np.median(cagr),
           "CAGR 5th pct": np.percentile(cagr, 5), "CAGR 95th pct": np.percentile(cagr, 95),
           "Max DD median": np.median(dd), "Max DD worst 5%": np.percentile(dd, 5),
           "Prob. of net loss": (term < 1).mean()}
    if benchmark_mult is not None: out["Prob. beat benchmark"] = (term > benchmark_mult).mean()
    return out

strat_rets = curve.pct_change().dropna()
bench_mult = bench_curve.iloc[-1] / bench_curve.iloc[0]
paths = monte_carlo(strat_rets, n_sims=5000, method="block", seed=42)
summary = mc_summary(paths, benchmark_mult=bench_mult)

print("MONTE CARLO (block bootstrap, 5000 sims) — distribution of outcomes\n")
for k, v in summary.items():
    if "Prob" in k:        print(f"  {k:<26} {v*100:5.1f}%")
    elif "CAGR" in k or "DD" in k: print(f"  {k:<26} {v*100:+5.1f}%")
    else:                  print(f"  {k:<26} {v:5.2f}x")

In [ ]:
# Fan chart: percentile bands of simulated equity paths
pct = {p: np.percentile(paths, p, axis=0) for p in [5, 25, 50, 75, 95]}
x = np.arange(paths.shape[1])
plt.figure(figsize=(11, 5.5))
plt.fill_between(x, pct[5], pct[95], color="#2a9d8f", alpha=0.12, label="5–95th pct")
plt.fill_between(x, pct[25], pct[75], color="#2a9d8f", alpha=0.25, label="25–75th pct")
plt.plot(x, pct[50], color="#1b7f4b", lw=2, label="median path")
plt.plot(x, np.cumprod(np.r_[1, 1 + strat_rets.values]), color="#e76f51", lw=1.5,
         ls="--", label="actual backtest")
plt.axhline(bench_mult, color="#9aa6b2", ls=":", lw=1.4, label=f"benchmark terminal ({bench_mult:.2f}x)")
plt.yscale("log"); plt.xlabel("months"); plt.ylabel("growth of $1 (log)")
plt.title("Monte Carlo outcome cone vs. the single actual backtest"); plt.legend(); plt.show()

print(f"Your actual backtest ({curve.iloc[-1]:.2f}x) is just ONE draw from this cone.")
print(f"In {summary['Prob. of net loss']*100:.0f}% of simulations the strategy ended below break-even.")
if "Prob. beat benchmark" in summary:
    print(f"It beat buy-and-hold in {summary['Prob. beat benchmark']*100:.0f}% of simulations.")

In [ ]:
# Decay-adjusted FORWARD scenario (not a forecast): recenter mean to the
# publication-decay expectation from Section 9, keep the volatility.
def decay_adjusted_returns(returns, excess_haircut, bench_periodic):
    r = pd.Series(returns).dropna()
    excess = r.mean() - bench_periodic
    target = bench_periodic + excess * (1 - excess_haircut)
    return (r - r.mean() + target).values

bench_periodic = bench_curve.pct_change().dropna().mean()
adj = decay_adjusted_returns(strat_rets, excess_haircut=0.58, bench_periodic=bench_periodic)
fwd = monte_carlo(adj, n_sims=5000, method="block", seed=7)
fwd_sum = mc_summary(fwd, benchmark_mult=(1 + bench_periodic) ** (len(strat_rets)) )

print("DECAY-ADJUSTED FORWARD SCENARIO (58% haircut on excess return — Section 9)\n")
print("  This is what the future could look like IF the public-factor decay holds.")
print("  It is a risk scenario, not a prediction.\n")
print(f"  Realized monthly mean : {strat_rets.mean()*100:+.2f}%")
print(f"  Haircut monthly mean  : {np.mean(adj)*100:+.2f}%  (volatility unchanged)")
print(f"  Forward CAGR median   : {fwd_sum['CAGR median']*100:+.1f}%")
print(f"  Forward 5th-pct CAGR  : {fwd_sum['CAGR 5th pct']*100:+.1f}%")
print(f"  Forward prob. of loss : {fwd_sum['Prob. of net loss']*100:.0f}%")

## 11 · Machine learning — *does a model actually beat the simple rule?*

The factor composite so far is a **fixed linear rule**: add up four z-scores. Machine learning asks whether a model that *learns* from history — including **nonlinear interactions** between factors — predicts better. This is the canonical setup from Gu, Kelly & Xiu (2020, *Review of Financial Studies*), who found tree- and network-based models delivered real gains in cross-sectional return prediction, with the edge coming specifically from nonlinear interactions that linear models miss.

Two things make this honest rather than a backtest-overfitting machine:

- **Leakage-safe validation.** We use a **purged, walk-forward** scheme (López de Prado's discipline): train only on months strictly before the test month, with a one-month **embargo** so overlapping label windows can't leak the future into the past. Random k-fold cross-validation — the default everywhere else — would quietly let the model peek ahead and produce fake skill. We never do that here.
- **A fair benchmark: the simple rule itself.** The ML models must beat the plain linear composite **out-of-sample**, not just look good. We report out-of-sample AUC for each.

> **The scale caveat that matters most.** Gu-Kelly-Xiu used ~30,000 stocks. Your universe is ~30. That is *orders of magnitude* too small for ML to reliably shine — with this little data, a flexible model is far more likely to overfit than to find real nonlinear structure. So the honest expected outcome is that ML roughly **matches or slightly trails** the simple composite. If it does, that's not failure — it's the correct, valuable finding that the simple rule is good enough and ML adds fragility, not edge, at your scale. The notebook computes the real answer below.

In [ ]:
# ML needs scikit-learn (pre-installed in Colab). Uncomment if missing:
# !pip install scikit-learn --quiet
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

FEATURES = ["z_momentum", "z_trend", "z_quality", "z_value"]
W = CONFIG["weights"]

def build_panel(prices, cfg, start=None):
    """One row per (month, stock): factor z-scores as-of t, and whether the stock
       BEAT the cross-sectional median return next month (the label). No lookahead."""
    pme = prices.resample(cfg["rebalance"]).last(); me = pme.index
    if start: me = me[me >= pd.Timestamp(start)]
    rows = []
    for i in range(len(me) - 1):
        asof, nxt = me[i], me[i + 1]
        if prices.loc[:asof].shape[0] < 260: continue
        f = compute_scores(prices, asof, cfg)
        if f.empty: continue
        fwd = (pme.loc[nxt] / pme.loc[asof] - 1).reindex(f.index)
        med = fwd.median()
        for tk in f.index:
            if pd.notna(fwd[tk]):
                rows.append({"month": asof, "ticker": tk,
                             **{k: f.loc[tk, k] for k in FEATURES},
                             "composite": f.loc[tk, "composite"],
                             "fwd": fwd[tk], "label": int(fwd[tk] > med)})
    return pd.DataFrame(rows)

def walk_forward(panel, model_fn, embargo=1):
    """Purged, expanding-window OOS predictions. Train < (test month - embargo)."""
    months = sorted(panel["month"].unique()); oos = []
    for i in range(len(months)):
        test_m = months[i]; train_m = months[:max(0, i - embargo)]
        if len(train_m) < 6: continue
        tr = panel[panel.month.isin(train_m)]; te = panel[panel.month == test_m]
        if te.empty or tr.label.nunique() < 2: continue
        sc = StandardScaler().fit(tr[FEATURES])
        m = model_fn(); m.fit(sc.transform(tr[FEATURES]), tr.label)
        p = m.predict_proba(sc.transform(te[FEATURES]))[:, 1]
        for tk, prob, lab, fwd, comp in zip(te.ticker, p, te.label, te.fwd, te.composite):
            oos.append({"month": test_m, "ticker": tk, "prob": prob,
                        "label": lab, "fwd": fwd, "composite": comp})
    return pd.DataFrame(oos)

panel = build_panel(prices, CONFIG, start=start_date)
print(f"Panel: {len(panel)} rows | {panel.month.nunique()} months | "
      f"{panel.ticker.nunique()} stocks | base rate {panel.label.mean():.1%}")

models = {"Logistic regression": lambda: LogisticRegression(max_iter=300, C=1.0),
          "Gradient boosting":   lambda: HistGradientBoostingClassifier(
                                     max_depth=3, max_iter=200, learning_rate=0.05,
                                     l2_regularization=1.0)}
oos_preds = {}
print("\nOUT-OF-SAMPLE skill (purged walk-forward) — AUC 0.50 = coin flip:\n")
for name, fn in models.items():
    oos = walk_forward(panel, fn, embargo=1); oos_preds[name] = oos
    print(f"  {name:<22} OOS AUC = {roc_auc_score(oos.label, oos.prob):.3f}")
print(f"  {'Simple composite rule':<22} OOS AUC = {roc_auc_score(panel.label, panel.composite):.3f}  (no learning)")

In [ ]:
# Does the ML ranking beat the simple-composite ranking as a PORTFOLIO (out-of-sample)?
def oos_portfolio(df, rank_col, top_n):
    eq, dates, vals = 1.0, [], []
    for m, g in df.groupby("month"):
        picks = g.nlargest(min(top_n, len(g)), rank_col)
        eq *= (1 + picks["fwd"].mean())          # equal-weight top-N, realized next month
        dates.append(m); vals.append(eq)
    return pd.Series(vals, index=pd.DatetimeIndex(dates))

ref = list(oos_preds.values())[0][["month", "ticker", "composite", "fwd"]].copy()
comp_curve_oos = oos_portfolio(ref, "composite", CONFIG["top_n"])
print("OUT-OF-SAMPLE portfolio: ML ranking vs the simple composite ranking\n")
rows = {"Simple composite": metrics(comp_curve_oos)}
for name, oos in oos_preds.items():
    rows[name] = metrics(oos_portfolio(oos, "prob", CONFIG["top_n"]))
tbl = pd.DataFrame(rows).T
for c in ["CAGR", "Vol", "MaxDD"]: tbl[c] = tbl[c].map(lambda x: f"{x*100:+.1f}%")
tbl["HitRate"] = tbl["HitRate"].map(lambda x: f"{x*100:.0f}%")
tbl[["Sharpe", "Sortino"]] = tbl[["Sharpe", "Sortino"]].round(2)
display(tbl[["CAGR", "Vol", "Sharpe", "Sortino", "MaxDD", "HitRate"]]) if "display" in dir() else print(tbl)

# Feature importance (gradient boosting, permutation — honest, model-agnostic)
sc = StandardScaler().fit(panel[FEATURES])
gbm = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05,
                                     l2_regularization=1.0).fit(sc.transform(panel[FEATURES]), panel.label)
imp = permutation_importance(gbm, sc.transform(panel[FEATURES]), panel.label,
                             n_repeats=10, random_state=0)
fi = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False)
print("\nPermutation importance (which factor the model leans on most):")
for k, v in fi.items(): print(f"  {k:<12} {v:+.4f}")

### How to read this honestly

- **If ML's OOS AUC is ~0.50 and it doesn't beat the composite portfolio** — the most likely result at this universe size — then the simple linear rule wins. That's a real finding: it says your edge (if any) is linear and small, and adding a model just adds overfitting risk. Keep it simple.
- **If a model clears the composite by a clear margin out-of-sample**, look hard at *why* before believing it: re-run with a different `start_date` and a larger embargo, and check the feature importance makes economic sense. One lucky split is not skill.
- **Never** judge these models in-sample. The whole point of the purged walk-forward is that in-sample accuracy is meaningless here — a flexible model can memorize 30 stocks trivially.

This mirrors the discipline of the rest of the notebook: the tools are built to tell you when something *doesn't* work, cheaply, before any capital is involved. A good quant treats "the simple model was better" as a perfectly good — often the most common — answer.

## 12 · Read this before you act

**What can make these results lie to you:**

- **Survivorship & selection bias.** You hand-picked the universe today, knowing which themes did well. That alone can inflate the backtest. The honest test is whether the *ranking logic* adds value beyond just owning the basket equally.
- **No costs modeled by default.** Commissions are ~zero at most brokers now, but **slippage and taxes are real**, and the turnover figure above tells you how often you'd trade.
- **One history is one sample.** An 8-year backtest covers *one* path. Change the start date or the weights and the numbers move — if they move *a lot*, the edge is fragile.
- **Overfitting.** If you tune the weights until the curve looks great, you've fit the past, not found the future. Pick a thesis, then test it — not the other way around.

**The most valuable outcome:** if the strategy does **not** clearly beat buy-and-hold after costs, that's a genuine finding — it means owning a low-cost index (or your equal-weighted impact basket) is the smarter play, and you just saved yourself the effort and risk. A good analyst is as happy to kill a strategy as to keep one.

**Sensible next steps:** compare against an **equal-weight** version of your own universe (not just SPY); test 2–3 different start dates; paper-trade the live picks for a few months before any real capital. This is not financial advice.

In [ ]:
# Sanity benchmark: does the RANKING beat just owning your basket equally?
ew = prices.resample(CONFIG["rebalance"]).last().pct_change().dropna()
ew_curve = (1 + ew.mean(axis=1)).cumprod()
ew_curve = ew_curve.reindex(curve.index).ffill(); ew_curve /= ew_curve.iloc[0]
me_ew = metrics(ew_curve)
print("Does stock-picking add value over equal-weighting your own universe?\n")
out = pd.DataFrame({"Strategy (ranked)": metrics(curve),
                    "Equal-weight basket": me_ew,
                    f"Buy & hold {BENCHMARK}": metrics(bench_curve)}).T
for c in ["CAGR","Vol","MaxDD"]: out[c] = out[c].map(lambda x: f"{x*100:.1f}%")
out["HitRate"] = out["HitRate"].map(lambda x: f"{x*100:.0f}%")
out[["Sharpe","Sortino"]] = out[["Sharpe","Sortino"]].round(2)
out[["CAGR","Vol","Sharpe","Sortino","MaxDD","HitRate"]]

---
### Switching to *broad market + exclusions* instead of a curated list

If you'd rather not hand-pick names, replace the universe in Section 1 with a broad set (e.g. S&P 500 constituents) and apply an **exclusion** filter on industry. You can pull each ticker's industry via `yf.Ticker(t).info.get("industry")` and drop categories you object to — tobacco, defense/weapons, gambling, thermal coal, etc. The rest of the notebook runs unchanged. Note `.info` is slow and occasionally missing fields, so cache it and handle gaps gracefully.

*Built as research tooling. Not investment advice. No guarantee of future results.*